In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('CSC_CIC_2018Final.csv')

In [3]:
df.duplicated().sum()

np.int64(433195)

In [4]:
global_dups = df.duplicated().sum()

label_dups = df.groupby("Label").apply(
    lambda x: x.duplicated().sum()
).sum()

print(global_dups, label_dups)


433195 433195


C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_6464\3639644098.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  label_dups = df.groupby("Label").apply(


In [5]:
print(df["Label"].isna().sum())

0


In [6]:
# Use dropna=False to include the rows with missing labels
df.groupby("Label", dropna=False).apply(lambda x: x.duplicated().sum())

C:\Users\Cloud-2\AppData\Local\Temp\ipykernel_6464\2161749747.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Label", dropna=False).apply(lambda x: x.duplicated().sum())


Label
Benign                       39210
Bot                           3881
Brute Force -Web                 0
Brute Force -XSS                 0
DDOS attack-HOIC             17551
DDOS attack-LOIC-UDP             0
DDoS attacks-LOIC-HTTP          16
DoS attacks-GoldenEye           53
DoS attacks-Hulk             27039
DoS attacks-SlowHTTPTest    120428
DoS attacks-Slowloris          705
FTP-BruteForce              154008
Infilteration                   37
SQL Injection                    0
SSH-Bruteforce               70267
dtype: int64

In [8]:
df.shape

(16232943, 80)

In [9]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 59721


In [10]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 59721


In [11]:
# This will show only the columns that have at least one NaN
print(df.columns[df.isna().any()].tolist())

['Flow Byts/s']


In [12]:
df['Flow Byts/s'].isna().sum()

np.int64(59721)

In [13]:
# 1. Filter the dataframe for only rows where Flow Bytes/s is NaN
nan_data = df[df['Flow Byts/s'].isna()]

# 2. Count the labels within that subset
distribution = nan_data['Label'].value_counts()

print("Distribution of NaNs per Label:")
print(distribution)

# 3. Optional: See the percentage of each label that is "broken"
total_per_label = df['Label'].value_counts()
nan_percentage = (distribution / total_per_label) * 100
print("\nPercentage of each Label that has NaNs:")
print(nan_percentage.dropna())

Distribution of NaNs per Label:
Label
Benign            58877
Infilteration       838
FTP-BruteForce        6
Name: count, dtype: int64

Percentage of each Label that has NaNs:
Label
Benign            0.436621
FTP-BruteForce    0.003103
Infilteration     0.517495
Name: count, dtype: float64


In [14]:
df['Label'].value_counts()

Label
Benign                      13484708
DDOS attack-HOIC              686012
DDoS attacks-LOIC-HTTP        576191
DoS attacks-Hulk              461912
Bot                           286191
FTP-BruteForce                193360
SSH-Bruteforce                187589
Infilteration                 161934
DoS attacks-SlowHTTPTest      139890
DoS attacks-GoldenEye          41508
DoS attacks-Slowloris          10990
DDOS attack-LOIC-UDP            1730
Brute Force -Web                 611
Brute Force -XSS                 230
SQL Injection                     87
Name: count, dtype: int64

In [15]:
# 1. Select only the columns that contain text/objects
string_columns = df.select_dtypes(include=['object']).columns

# 2. Check if the stripped string is empty
# .str.strip() removes spaces; .eq('') checks if it is then empty
blanks_mask = df[string_columns].apply(lambda x: x.str.strip().eq('')).any(axis=1)

total_blank_rows = blanks_mask.sum()
print(f"Total rows with at least one blank entry: {total_blank_rows}")

Total rows with at least one blank entry: 0


In [16]:
import numpy as np

# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Total number of infinite values
total_inf = np.isinf(numeric_df).values.sum()

print(f"Total infinite entries: {total_inf}")

Total infinite entries: 131799


In [17]:
# Check if any value in a row is infinite
inf_rows = np.isinf(numeric_df).any(axis=1).sum()

print(f"Total rows with at least one infinite value: {inf_rows}")

Total rows with at least one infinite value: 95760


In [18]:
import numpy as np

# Select only numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Count infinite values (positive and negative) per column
inf_counts = np.isinf(df[numeric_cols]).sum()

# Display only columns that have at least one infinite value
print("Infinite values per column:")
print(inf_counts[inf_counts > 0])

Infinite values per column:
Flow Pkts/s    95760
Flow Byts/s    36039
dtype: int64


In [19]:
# 1. Create a boolean mask for any row that has at least one infinite value
inf_rows_mask = np.isinf(df[numeric_cols]).any(axis=1)

# 2. Filter the dataframe using that mask and count the Labels
inf_label_dist = df[inf_rows_mask]['Label'].value_counts()

print("\nDistribution of Infinite entries per Label:")
print(inf_label_dist)


Distribution of Infinite entries per Label:
Label
Benign            94459
Infilteration      1295
FTP-BruteForce        6
Name: count, dtype: int64


In [20]:
total_inf_cells = np.isinf(df[numeric_cols]).values.sum()
total_rows_with_inf = inf_rows_mask.sum()

print(f"\nTotal infinite cells in dataset: {total_inf_cells}")
print(f"Total rows affected by infinity: {total_rows_with_inf}")


Total infinite cells in dataset: 131799
Total rows affected by infinity: 95760


In [21]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [22]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 95760


In [23]:
total_nans = df.isna().sum().sum()
print(f"Total missing values in DF: {total_nans}")

Total missing values in DF: 191520


In [24]:
# This will show only the columns that have at least one NaN
print(df.columns[df.isna().any()].tolist())

['Flow Pkts/s', 'Flow Byts/s']


In [25]:
# Save the cleaned DataFrame back to your file
df.to_csv('CSC_CIC_2018Final.csv', index=False)

In [26]:
pd.set_option('display.max_rows', None)

# 2. Run your zero-count check again
# Replace 'df' with the name of your dataframe (e.g., X_normalized or data)
zero_counts = (df == 0).sum()

# 3. Print the result
print(zero_counts)

Fwd Pkts/s              95774
Active Min           14605586
Protocol               238313
Flow Pkts/s                 0
Bwd PSH Flags        16232943
Subflow Fwd Pkts            0
Idle Min             13885518
Bwd Header Len        4148558
Active Mean          14605586
Bwd URG Flags        16232943
Bwd Pkt Len Min      12019933
Active Std           15107385
Bwd IAT Max           9337704
Fwd Pkts/b Avg       16232943
CWE Flag Count       16230284
Fwd Pkt Len Min      11996471
Init Fwd Win Byts       77743
Bwd IAT Mean          9337704
Pkt Size Avg          5026253
Idle Std             14945289
Flow Duration           95760
Subflow Bwd Byts      6007265
Fwd Act Data Pkts     8675488
Flow IAT Mean           95760
PSH Flag Cnt          9868262
Fwd URG Flags        16230284
Bwd Blk Rate Avg     16232943
Pkt Len Mean          5026253
Idle Mean            13885518
Bwd Pkts/s            4138946
Init Bwd Win Byts     1114835
Flow IAT Min           883358
FIN Flag Cnt         16155099
Fwd Pkt Le

In [27]:
pd.set_option('display.max_rows', None)

# 2. Calculate the percentage of zeros
# (df_test_new == 0).mean() gives the proportion, * 100 gives the %
zero_percentage = (df == 0).mean() * 100

# 3. Print the result (sorted descending so you see the "emptiest" columns first)
print(zero_percentage)

Fwd Pkts/s             0.589998
Active Min            89.974972
Protocol               1.468083
Flow Pkts/s            0.000000
Bwd PSH Flags        100.000000
Subflow Fwd Pkts       0.000000
Idle Min              85.539129
Bwd Header Len        25.556413
Active Mean           89.974972
Bwd URG Flags        100.000000
Bwd Pkt Len Min       74.046542
Active Std            93.066211
Bwd IAT Max           57.523174
Fwd Pkts/b Avg       100.000000
CWE Flag Count        99.983620
Fwd Pkt Len Min       73.902009
Init Fwd Win Byts      0.478921
Bwd IAT Mean          57.523174
Pkt Size Avg          30.963289
Idle Std              92.067649
Flow Duration          0.589912
Subflow Bwd Byts      37.006629
Fwd Act Data Pkts     53.443716
Flow IAT Mean          0.589912
PSH Flag Cnt          60.791577
Fwd URG Flags         99.983620
Bwd Blk Rate Avg     100.000000
Pkt Len Mean          30.963289
Idle Mean             85.539129
Bwd Pkts/s            25.497200
Init Bwd Win Byts      6.867732
Flow IAT

In [2]:
import pandas as pd
df = pd.read_csv("CSC_CIC_2018Final.csv")

In [ ]:
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

df.to_csv('CSC_CIC_2018Final_binary.csv', index=False)